# Milestone 4 lab, notebook 2 of 7: Your first model

## Where we are

| Notebook | What you do |
|---|---|
| 1. Load first, transform later | Python stops transforming. Compare ELT with milestone 1's ETL |
| **2. Your first model** (you are here) | Write one SELECT in a file, and let dbt build it |
| 3. Clean, join, enrich | One row per order again, then two joins and their traps |
| 4. A loop or a set | Answer one question row by row, then with one statement |
| 5. Layers and the incremental model | Three layers, and a report that rebuilds only its newest days |
| 6. Tests on the tables | Break the data on purpose, and watch the pipeline refuse it |
| 7. The production version | One morning as ELT: Python loads, dbt transforms and tests |

One command built seven tables and views in notebook 1. This notebook opens the box. You write the first transformation by hand, and then see the same thing as a dbt **model**.

In [1]:
from lab_helpers import MILESTONE_DIR, dbt, lab_warehouse, load_only, show_doc, table

con = lab_warehouse(fresh=True)
load_only(con, "2026-06-08", "2026-06-09")

Landed 226 raw rows for the morning of 2026-06-08.
Landed 102 raw rows for the morning of 2026-06-09.


## Step 1: Raw data is not ready to use

Here are three raw rows. Try to add up their totals.

In [2]:
table(con, "SELECT landing_id, payload->>'order_id' AS order_id, payload->>'status' AS status, payload->>'total_cents' AS total_cents, payload->>'placed_at' AS placed_at FROM raw.orders_landing LIMIT 3")

,landing_id,order_id,status,total_cents,placed_at
0,1,1001,delivered,3276,2026-06-01T12:10:06Z
1,2,1002,delivered,2512,2026-06-01T13:17:46Z
2,3,1004,delivered,2721,2026-06-01T13:46:54Z


In [3]:
import psycopg

try:
    con.execute("SELECT sum(payload->>'total_cents') FROM raw.orders_landing")
except psycopg.Error as error:
    print("Postgres refused:", error.diag.message_primary)

Postgres refused: function sum(text) does not exist


Everything that comes out of the JSON is **text**. `"3276"` is four characters, not a number. You cannot add text.

## Step 2: Type casting, by hand

Changing a value from one type to another is called a **cast**. In Postgres you write it with `::`.

In [4]:
table(con, """
    SELECT (payload->>'order_id')::bigint          AS order_id,
           (payload->>'total_cents')::bigint       AS total_cents,
           (payload->>'placed_at')::timestamptz    AS placed_at
    FROM raw.orders_landing LIMIT 3
""")

,order_id,total_cents,placed_at
0,1001,3276,2026-06-01 12:10:06+00:00
1,1002,2512,2026-06-01 13:17:46+00:00
2,1004,2721,2026-06-01 13:46:54+00:00


In [5]:
table(con, "SELECT sum((payload->>'total_cents')::bigint) AS total_cents FROM raw.orders_landing WHERE run_id = '2026-06-08_1'")

,total_cents
0,549830


Now it adds up.

## Step 3: Cleaning, by hand

**Cleaning** makes each value valid: the right type, one spelling, a real meaning. The order system is tidy today. To see why you clean anyway, land one row the way a messy source might send it.

In [6]:
con.execute("""
    INSERT INTO raw.orders_landing (payload, run_id, page_no, loaded_at)
    SELECT payload || '{"status": " Delivered ", "city": "ngt", "updated_at": "2026-06-09T05:00:00Z"}', 'messy', 1, now()
    FROM raw.orders_landing WHERE (payload->>'order_id')::bigint = 1130 ORDER BY landing_id DESC LIMIT 1
""")

table(con, """
    SELECT payload->>'status' AS status_as_sent, count(*) AS rows
    FROM raw.orders_landing GROUP BY 1 ORDER BY 2 DESC
""")

,status_as_sent,rows
0,delivered,316
1,placed,10
2,out_for_delivery,1
3,cancelled,1
4,Delivered,1


`" Delivered "` with a capital letter and two spaces is a different value from `"delivered"`. A count by status would now have one group too many. One rule fixes every such row:

In [7]:
table(con, """
    SELECT lower(trim(payload->>'status')) AS status, count(*) AS rows
    FROM raw.orders_landing GROUP BY 1 ORDER BY 2 DESC
""")

,status,rows
0,delivered,317
1,placed,10
2,out_for_delivery,1
3,cancelled,1


Clean each column **once**, in one place. Every table built after that can trust it.

## Step 4: The same thing, as a model

You wrote two `SELECT` statements. A dbt **model** is exactly that: one `SELECT`, saved in a file. Here is the real one.

In [8]:
print((MILESTONE_DIR / "dbt_project" / "models" / "staging" / "stg_orders.sql").read_text())

-- Staging: the raw orders, cleaned, typed, and with ONE row per order.
-- Nothing is counted or added up here.

with landed as (

    select landing_id, run_id, loaded_at, payload
    from {{ source('raw', 'orders_landing') }}

),

typed as (   -- type casting: every value out of the JSON gets its real type

    select
        landing_id,
        run_id,
        loaded_at,
        (payload->>'order_id')::bigint                      as order_id,
        (payload->>'customer_id')::bigint                   as customer_id,
        upper(trim(payload->>'city'))                       as city,      -- cleaning: one spelling
        lower(trim(payload->>'status'))                     as status,    -- cleaning: one spelling
        (payload->>'placed_at')::timestamptz                as placed_at,
        (payload->>'updated_at')::timestamptz               as updated_at,
        (payload->>'delivered_at')::timestamptz             as delivered_at,
        (payload->>'cancelled_at')::timestamptz 

Find your two ideas in it: the casts, and `lower(trim(...))`. Three things are new.

| In the file | Meaning |
|---|---|
| `{{ source('raw', 'orders_landing') }}` | "The raw landing table." dbt writes the real name. The table is described in `models/sources.yml` |
| `with ... as ( ... )` | A named step. The model reads top to bottom: `landed`, then `typed`, then `ranked` |
| `{{ var("business_time_zone") }}` | A setting kept in one place, `dbt_project.yml` |

There is no `CREATE TABLE` and no `INSERT` anywhere. You say **what the table is**. dbt decides how to build it.

## Step 5: Build only this model

In [9]:
dbt("run", "--select", "stg_orders")

1 of 1 OK created sql view model staging.stg_orders ............................ [CREATE VIEW]
Completed successfully
Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1


In [10]:
table(con, "SELECT order_id, city, status, order_date, total_cents, updated_at, run_id FROM staging.stg_orders WHERE order_id IN (1130, 1196) ORDER BY 1")

,order_id,city,status,order_date,total_cents,updated_at,run_id
0,1130,NGT,delivered,2026-06-05,1475,2026-06-09 05:00:00+00:00,messy
1,1196,RVB,delivered,2026-06-06,1796,2026-06-07 14:20:05+00:00,2026-06-09_1


Order 1130 is one row. Its status is `delivered`, clean, and its city is `NGT`. The messy row was the newest version, and the model cleaned it.

Order 1196, the two-minutes-to-midnight order, is on Saturday 6 June. The time zone rule of milestone 1 lives in this file now, in one line.

Look at what dbt really ran. It keeps the finished SQL in a `target` folder.

In [11]:
compiled = (MILESTONE_DIR / "dbt_project" / "target" / "compiled" / "arden" / "models" / "staging" / "stg_orders.sql").read_text()
print(compiled[compiled.index("with landed"):compiled.index("typed as")])

with landed as (

    select landing_id, run_id, loaded_at, payload
    from "arden_m4_lab"."raw"."orders_landing"

),




`{{ source(...) }}` became a real table name. dbt then wrapped the whole `SELECT` in `create view ... as`.

In [12]:
con.execute("DELETE FROM raw.orders_landing WHERE run_id = 'messy'")
con.close()
show_doc("LLD.md", "2. A model")

**From `docs/LLD.md`**

> ## 2. A model
> 
> A **model** is one `SELECT` in a file. dbt builds a table or a view from it. A model never says `CREATE TABLE` or `INSERT`.
> 
> ```sql
> -- models/core/order_items.sql
> select
>     i.order_id,
>     i.line_no,
>     i.product_name,
>     i.quantity,
>     i.unit_price_cents,
>     case when i.item_status = 'ok' then i.quantity * i.unit_price_cents else 0 end as line_total_cents
> from {{ ref('stg_order_items') }} as i
> ```
> 
> | Piece | Meaning |
> |---|---|
> | `{{ ref('stg_order_items') }}` | "The model called `stg_order_items`." dbt fills in the real table name, and learns that this model must be built after that one |
> | `{{ source('raw', 'orders_landing') }}` | A table dbt did not build: the raw landing table |
> | `{{ var('business_time_zone') }}` | A setting from `dbt_project.yml` |
> 
> From the `ref` calls dbt works out the order to build in:
> 
> *(There is a diagram here. Open `docs/LLD.md` to see it.)*

## New words

| Word | Meaning | Here |
|---|---|---|
| **Model** | "A model is one SELECT in a file, and dbt builds a table or a view from it." | `stg_orders.sql` |
| **Type casting** | Changing a value from one type to another. Text out of JSON becomes a number or a time | `(payload->>'total_cents')::bigint` |
| **Cleaning** | "Cleaning makes each value valid." A valid value has the right type, one standard spelling and a real meaning | `lower(trim(payload->>'status'))` |

The definitions are from the D8Loop chapter [Transformation Fundamentals](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/transformation-fundamentals).

## What you now know

- Values out of JSON are text. A cast gives each one its real type.
- Cleaning gives each value one spelling. Do it once, in the staging model.
- A model is one `SELECT` in a file. dbt builds the table or view.
- `source()` names a raw table. dbt fills in the real name.

**Say it in an interview:** "My staging models do three things and nothing else: cast every field out of the raw JSON to its real type, give each value one spelling, and keep one row per key. Nothing is added up there. Every model downstream reads from staging, so a value is cleaned in exactly one place."

Next: [notebook 3](03_clean_join_enrich.ipynb) finishes this model, and joins it to two other tables.